# Run preprocessing (CPU, cluster)

Runs the CPU preprocessing tier end-to-end: builds the cohort (both anchors), extracts ICD
timing, builds non-text covariates, renders the data-availability report, then tokenizes notes.
Data transfer is handled separately; this notebook stops after writing the token batches.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find v2 root from {start}")


V2_ROOT = find_v2_root()
print(f"v2 root: {V2_ROOT}")
print(f"Python:  {sys.executable}")

## Run stages

Each stage is invoked as a subprocess so a failure on one stage does not prevent inspecting
what already ran. `report_data_availability` renders the combination table inline (it prints
its summary JSON and writes the marginals/combinations/pairwise CSVs to `SURV_PATH`).

In [ ]:
STAGES = [
    ["pipelines.preprocessing.build_cohort"],
    ["pipelines.preprocessing.extract_ICD_times"],
    ["pipelines.preprocessing.generate_all_non_text_covariates"],
    ["pipelines.preprocessing.text_preprocessing_and_tokenization"],
]


def run_stage(args: list[str]) -> None:
    print("\n=== " + " ".join(args) + " ===", flush=True)
    subprocess.run([sys.executable, "-m", *args], cwd=V2_ROOT, check=True)


for args in STAGES:
    run_stage(args)

print(f"\nDone. {len(STAGES)} preprocessing stages succeeded.")